In [1]:
# =========================================================
# 01. 라이브러리 불러오기
# =========================================================

import pandas as pd
import ast


# =========================================================
# 02. 파일 경로 설정
# =========================================================

INPUT_PATH = "../../../../data/preprocessed/final_review_categories.csv"
SAVE_PATH = "../../../../data/preprocessed/tableau_issue_long.csv"


# =========================================================
# 03. 데이터 불러오기
# =========================================================

df = pd.read_csv(INPUT_PATH)

print("원본 데이터 크기:", df.shape)
display(df.head())


# =========================================================
# 04. Final Categories 컬럼명 확인
# =========================================================
# Tableau에서 보이는 이름은 Final Categories지만,
# CSV 실제 컬럼명은 final_categories일 가능성이 큼.
# 아래 출력으로 정확한 컬럼명을 먼저 확인한다.

print(df.columns.tolist())

원본 데이터 크기: (72268, 33)


,Unnamed: 0,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,...,playtime_last_two_weeks_hours,playtime_at_review_hours,primary_genre,sentiment,clean_review,issue_categories,index,llm_categories,llm_reason,final_categories
0,0,18699465,324470,english,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",1445885616,1445885616,True,1,0,...,0.0,0.216667,Racing,positive,bgm good graphics good but slip effect is too ...,"['난이도/밸런스', '아트/비주얼']",1,NaN,NaN,"['난이도/밸런스', '아트/비주얼']"
1,1,18699648,324470,english,"this game is like a zen-garden, I love it! \n\...",1445886217,1482541338,True,5,0,...,0.0,12.666667,Racing,positive,this game is like a zen garden i love it pros ...,"['업데이트/개발', '조작/UX', '음악/사운드', '분위기/감성']",2,NaN,NaN,"['업데이트/개발', '조작/UX', '음악/사운드', '분위기/감성']"
2,2,18700348,324470,english,Ever played Bhop? Surf? If so this games mecha...,1445889159,1445895001,True,16,0,...,0.0,0.916667,Racing,positive,ever played bhop surf if so this games mechani...,"['업데이트/개발', '아트/비주얼', '음악/사운드', '게임플레이']",3,NaN,NaN,"['업데이트/개발', '아트/비주얼', '음악/사운드', '게임플레이']"
3,3,18701774,324470,english,It's Lit,1445895144,1445895144,True,4,0,...,0.0,6.416667,Racing,positive,it s lit,['기타'],4,NaN,NaN,['기타']
4,4,18702904,324470,english,"Greatness comes in all sorts of things, but th...",1445899986,1445903284,True,3,1,...,0.0,0.650000,Racing,positive,greatness comes in all sorts of things but thi...,"['가격/가성비', '조작/UX']",6,NaN,NaN,"['가격/가성비', '조작/UX']"


['Unnamed: 0', 'recommendationid', 'appid', 'language', 'review', 'timestamp_created', 'timestamp_updated', 'voted_up', 'votes_up', 'votes_funny', 'weighted_vote_score', 'comment_count', 'steam_purchase', 'received_for_free', 'written_during_early_access', 'author_steamid', 'author_num_games_owned', 'author_num_reviews', 'author_last_played', 'created_date', 'updated_date', 'author_last_played_date', 'playtime_forever_hours', 'playtime_last_two_weeks_hours', 'playtime_at_review_hours', 'primary_genre', 'sentiment', 'clean_review', 'issue_categories', 'index', 'llm_categories', 'llm_reason', 'final_categories']


In [2]:
# =========================================================
# 05. 리스트 문자열을 실제 리스트로 바꾸는 함수
# =========================================================

def parse_category_list(x):
    """
    목적:
        문자열로 저장된 카테고리 리스트를 실제 파이썬 list로 바꾼다.

    예시 입력:
        "['가격/가성비', '난이도/밸런스']"

    예시 출력:
        ['가격/가성비', '난이도/밸런스']

    변환 실패 시:
        빈 리스트 [] 반환
    """

    if pd.isna(x):
        return []

    if isinstance(x, list):
        return x

    try:
        value = ast.literal_eval(str(x))

        if isinstance(value, list):
            return value
        else:
            return []

    except:
        return []


# =========================================================
# 06. final_categories를 리스트로 변환
# =========================================================

df["final_category_list"] = df["final_categories"].apply(parse_category_list)

print(df[["final_categories", "final_category_list"]].head())

                           final_categories               final_category_list
0                     ['난이도/밸런스', '아트/비주얼']                 [난이도/밸런스, 아트/비주얼]
1  ['업데이트/개발', '조작/UX', '음악/사운드', '분위기/감성']  [업데이트/개발, 조작/UX, 음악/사운드, 분위기/감성]
2  ['업데이트/개발', '아트/비주얼', '음악/사운드', '게임플레이']  [업데이트/개발, 아트/비주얼, 음악/사운드, 게임플레이]
3                                    ['기타']                              [기타]
4                       ['가격/가성비', '조작/UX']                   [가격/가성비, 조작/UX]


In [7]:
df["appid"].nunique()

187

In [ ]:
df

In [4]:
# =========================================================
# 07. 카테고리 1개당 1행으로 펼치기
# =========================================================

issue_long = df.explode("final_category_list").copy()

# 컬럼명 정리
issue_long = issue_long.rename(columns={
    "final_category_list": "final_category"
})

# 빈 카테고리 제거
issue_long = issue_long[
    issue_long["final_category"].notna()
].copy()

issue_long["final_category"] = issue_long["final_category"].astype("string").str.strip()

# 빈 문자열 제거
issue_long = issue_long[
    issue_long["final_category"] != ""
].copy()

print("변환 후 데이터 크기:", issue_long.shape)
display(issue_long.head())

변환 후 데이터 크기: (121354, 34)


,Unnamed: 0,recommendationid,appid,language,review,timestamp_created,timestamp_updated,voted_up,votes_up,votes_funny,...,playtime_at_review_hours,primary_genre,sentiment,clean_review,issue_categories,index,llm_categories,llm_reason,final_categories,final_category
0,0,18699465,324470,english,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",1445885616,1445885616,True,1,0,...,0.216667,Racing,positive,bgm good graphics good but slip effect is too ...,"['난이도/밸런스', '아트/비주얼']",1,NaN,NaN,"['난이도/밸런스', '아트/비주얼']",난이도/밸런스
0,0,18699465,324470,english,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",1445885616,1445885616,True,1,0,...,0.216667,Racing,positive,bgm good graphics good but slip effect is too ...,"['난이도/밸런스', '아트/비주얼']",1,NaN,NaN,"['난이도/밸런스', '아트/비주얼']",아트/비주얼
1,1,18699648,324470,english,"this game is like a zen-garden, I love it! \n\...",1445886217,1482541338,True,5,0,...,12.666667,Racing,positive,this game is like a zen garden i love it pros ...,"['업데이트/개발', '조작/UX', '음악/사운드', '분위기/감성']",2,NaN,NaN,"['업데이트/개발', '조작/UX', '음악/사운드', '분위기/감성']",업데이트/개발
1,1,18699648,324470,english,"this game is like a zen-garden, I love it! \n\...",1445886217,1482541338,True,5,0,...,12.666667,Racing,positive,this game is like a zen garden i love it pros ...,"['업데이트/개발', '조작/UX', '음악/사운드', '분위기/감성']",2,NaN,NaN,"['업데이트/개발', '조작/UX', '음악/사운드', '분위기/감성']",조작/UX
1,1,18699648,324470,english,"this game is like a zen-garden, I love it! \n\...",1445886217,1482541338,True,5,0,...,12.666667,Racing,positive,this game is like a zen garden i love it pros ...,"['업데이트/개발', '조작/UX', '음악/사운드', '분위기/감성']",2,NaN,NaN,"['업데이트/개발', '조작/UX', '음악/사운드', '분위기/감성']",음악/사운드


In [5]:
# =========================================================
# 08. Tableau에서 쓸 컬럼만 선택
# =========================================================

keep_cols = [
    "recommendationid",
    "appid",
    "name",
    "primary_genre",
    "sentiment",
    "review",
    "clean_review",
    "final_category",
    "llm_reason"
]

# 실제로 존재하는 컬럼만 선택
keep_cols = [col for col in keep_cols if col in issue_long.columns]

tableau_issue = issue_long[keep_cols].copy()

print("Tableau용 데이터 크기:", tableau_issue.shape)
display(tableau_issue.head())


# # =========================================================
# # 09. 저장
# # =========================================================

# tableau_issue.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")

# print("저장 완료:", SAVE_PATH)

Tableau용 데이터 크기: (121354, 8)


,recommendationid,appid,primary_genre,sentiment,review,clean_review,final_category,llm_reason
0,18699465,324470,Racing,positive,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",bgm good graphics good but slip effect is too ...,난이도/밸런스,NaN
0,18699465,324470,Racing,positive,"BGM - GOOD \nGRAPHICS - GOOD\n\nBut, Slip effe...",bgm good graphics good but slip effect is too ...,아트/비주얼,NaN
1,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! \n\...",this game is like a zen garden i love it pros ...,업데이트/개발,NaN
1,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! \n\...",this game is like a zen garden i love it pros ...,조작/UX,NaN
1,18699648,324470,Racing,positive,"this game is like a zen-garden, I love it! \n\...",this game is like a zen garden i love it pros ...,음악/사운드,NaN


In [6]:
# # =========================================================
# # 08. Tableau에서 쓸 컬럼만 선택
# # =========================================================

# keep_cols = [
#     "recommendationid",
#     "appid",
#     "name",
#     "primary_genre",
#     "sentiment",
#     "review",
#     "clean_review",
#     "final_category",
#     "llm_reason"
# ]

# # 실제로 존재하는 컬럼만 선택
# keep_cols = [col for col in keep_cols if col in issue_long.columns]

# tableau_issue = issue_long[keep_cols].copy()

# print("Tableau용 데이터 크기:", tableau_issue.shape)
# display(tableau_issue.head())


# # =========================================================
# # 09. 저장
# # =========================================================

# tableau_issue.to_csv(SAVE_PATH, index=False, encoding="utf-8-sig")

# print("저장 완료:", SAVE_PATH)